In [20]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

data = pd.read_csv("train.csv.gz")
y = data.SalePrice
x = data.drop(columns=["SalePrice", "Id"])

# 找出数值列和类别列
num_cols = x.select_dtypes(include=["int64", "float64"]).columns
str_cols = x.select_dtypes(include=["object"]).columns

# 数值列：用中位数填补缺失值
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# 类别列：先用众数填补缺失值，再做独热编码
str_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, num_cols),
        ("str", str_transformer, str_cols)
    ]
)

x = preprocessor.fit_transform(x)

print("Setup complete!")


Setup complete!


In [21]:
# 我们先进行预训练，找出最适合的参数，并引入 RMSLE 作为评估指标
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import root_mean_squared_error, root_mean_squared_log_error

param_dist = {
    "n_estimators": [200, 300, 500, 1000],
    "max_depth": [10, 20, 30, 50],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [1.0, "sqrt", "log2"]
}

search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=20,                         
    cv=5,                              
    scoring="neg_root_mean_squared_log_error",
    n_jobs=-1,
    random_state=42,
    verbose=1
)

# 在训练集上找最佳参数
search.fit(x, y)

print("最佳参数:", search.best_params_)
print("交叉验证最佳 RMSLE:", -search.best_score_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
最佳参数: {'n_estimators': 1000, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 1.0, 'max_depth': 20}
交叉验证最佳 RMSLE: 0.14574791767508583


In [22]:
# 开始正式训练模型
model = RandomForestRegressor(**search.best_params_, random_state=42, n_jobs=-1)
model.fit(x, y)

# 处理测试集数据
test_data = pd.read_csv("test.csv.gz")
test_ids = test_data["Id"] 
test_x = test_data.drop(columns=["Id"])
test_x = preprocessor.transform(test_x)
predicted_price = model.predict(test_x)

output = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": predicted_price
})
output.to_csv("output.csv", index=False)
print(output.head())
print("Successfully Predicted the House Price!!")

     Id      SalePrice
0  1461  128475.544791
1  1462  154833.010817
2  1463  181244.578902
3  1464  180929.658756
4  1465  199366.819345
Successfully Predicted the House Price!!
